In [1]:
import os
import random
import time
import gc
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

if 'strategy' not in locals():
    try:
        strategy = tf.distribute.MirroredStrategy()
        print(f"✅ Dual-GPU Strategy Initialized. Replicas: {strategy.num_replicas_in_sync}")
    except Exception as e:
        print(f"❌ GPU initialization failed: {e}")
        strategy = tf.distribute.get_strategy()
else:
    print(f"⚡ Strategy already active. Replicas: {strategy.num_replicas_in_sync}")

BATCH_SIZE = 32 * strategy.num_replicas_in_sync
AUTOTUNE = tf.data.AUTOTUNE
DATA_DIR = '/kaggle/input/datasets/uraninjo/augmented-alzheimer-mri-dataset-v2'

print("\nFetching Local File Paths...")
filepaths = []
for root, dirs, files in os.walk(DATA_DIR):
    for file in files:
        if file.lower().endswith(('.png', '.jpg', '.jpeg')):
            filepaths.append(os.path.join(root, file))

labels = [os.path.basename(os.path.dirname(path)) for path in filepaths]
df = pd.DataFrame({'filepath': filepaths, 'label': labels})
class_names = sorted(df['label'].unique())
class_indices = {name: idx for idx, name in enumerate(class_names)}
df['label_idx'] = df['label'].map(class_indices)

print(f"✅ Successfully pooled {len(df)} images.")
print(f"Classes found: {class_indices}")

def decode_image(filepath, label):
    image_bytes = tf.io.read_file(filepath)
    image = tf.io.decode_image(image_bytes, channels=3, expand_animations=False)
    image = tf.image.resize(image, [224, 224])
    image = preprocess_input(image)
    label = tf.one_hot(label, depth=len(class_names))
    return image, label

def decode_image_predict_only(filepath):
    image_bytes = tf.io.read_file(filepath)
    image = tf.io.decode_image(image_bytes, channels=3, expand_animations=False)
    image = tf.image.resize(image, [224, 224])
    image = preprocess_input(image)
    return image

def build_dataset(dataframe, is_training=False):
    dataset = tf.data.Dataset.from_tensor_slices((dataframe['filepath'].values, dataframe['label_idx'].values))
    dataset = dataset.map(decode_image, num_parallel_calls=AUTOTUNE)
    if is_training:
        dataset = dataset.shuffle(buffer_size=2048, seed=SEED)
    dataset = dataset.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return dataset

def build_predict_dataset(dataframe):
    dataset = tf.data.Dataset.from_tensor_slices(dataframe['filepath'].values)
    dataset = dataset.map(decode_image_predict_only, num_parallel_calls=AUTOTUNE, deterministic=True)
    dataset = dataset.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return dataset

def build_mobilenet_model():
    tf.keras.backend.clear_session()
    with strategy.scope():
        base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
        base_model.trainable = False
        x = GlobalAveragePooling2D()(base_model.output)
        x = Dense(256, activation='relu')(x)
        x = Dropout(0.5)(x)
        outputs = Dense(len(class_names), activation='softmax')(x)
        model = Model(inputs=base_model.input, outputs=outputs)
        model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
            loss='categorical_crossentropy',
            metrics=['accuracy']
        )
    return model

K_FOLDS = 5
EPOCHS = 50

skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)

fold_accuracies  = []
fold_f1_scores   = []
fold_precisions  = []
fold_recalls     = []
fold_conf_matrices = []

print("\n🚀 STARTING 5-FOLD CROSS-VALIDATION (MobileNetV2 on T4 x2 GPUs)...")
start_time = time.time()

for fold, (train_val_idx, test_idx) in enumerate(skf.split(df['filepath'], df['label_idx'])):
    fold_num = fold + 1
    print(f"\n{'='*50}\n🧪 STARTING FOLD {fold_num} / {K_FOLDS}\n{'='*50}")

    test_df      = df.iloc[test_idx]
    train_val_df = df.iloc[train_val_idx]
    train_df, val_df = train_test_split(
        train_val_df, test_size=0.1,
        stratify=train_val_df['label_idx'], random_state=SEED
    )

    train_ds   = build_dataset(train_df, is_training=True)
    val_ds     = build_dataset(val_df, is_training=False)
    test_ds    = build_dataset(test_df, is_training=False)
    predict_ds = build_predict_dataset(test_df)

    model = build_mobilenet_model()

    checkpoint_path = f'/kaggle/working/best_mobilenetv2_fold_{fold_num}.keras'
    checkpoint      = ModelCheckpoint(checkpoint_path, monitor='val_accuracy', save_best_only=True, mode='max', verbose=1)
    early_stopping  = EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True, verbose=1)

    print("\n⏳ Training Model...")
    history = model.fit(
        train_ds, validation_data=val_ds,
        epochs=EPOCHS, callbacks=[checkpoint, early_stopping], verbose=1
    )

    print(f"\n📊 Generating Training Curves for Fold {fold_num}...")
    plt.figure(figsize=(14, 5))
    
    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'], label='Training Accuracy', linewidth=2)
    plt.plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
    plt.title(f'Fold {fold_num}: Model Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend(loc='lower right')
    plt.grid(True, linestyle='--', alpha=0.7)

    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'], label='Training Loss', linewidth=2)
    plt.plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
    plt.title(f'Fold {fold_num}: Model Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend(loc='upper right')
    plt.grid(True, linestyle='--', alpha=0.7)

    plt.tight_layout()
    plt.savefig(f'/kaggle/working/training_curves_fold_{fold_num}.png', dpi=300)
    plt.close()

    print("\n🔍 Evaluating on unseen Pure Test Set...")
    model.load_weights(checkpoint_path)

    test_loss, test_acc = model.evaluate(test_ds, verbose=0)
    fold_accuracies.append(test_acc)

    true_labels = test_df['label_idx'].values
    pred_probs  = model.predict(predict_ds, verbose=0)
    pred_labels = np.argmax(pred_probs, axis=1)

    f1   = f1_score(true_labels, pred_labels, average='weighted')
    prec = precision_score(true_labels, pred_labels, average='weighted')
    rec  = recall_score(true_labels, pred_labels, average='weighted')
    cm   = confusion_matrix(true_labels, pred_labels)

    fold_f1_scores.append(f1)
    fold_precisions.append(prec)
    fold_recalls.append(rec)
    fold_conf_matrices.append(cm)

    print(f"\n📊 Generating Confusion Matrix for Fold {fold_num}...")
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=class_names, yticklabels=class_names)
    plt.title(f'Fold {fold_num}: Confusion Matrix')
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.tight_layout()
    plt.savefig(f'/kaggle/working/confusion_matrix_fold_{fold_num}.png', dpi=300)
    plt.close()

    print(f"✅ Fold {fold_num} | Acc: {test_acc*100:.2f}% | F1: {f1:.4f} | Precision: {prec:.4f} | Recall: {rec:.4f}")

    print("🧹 Wiping GPU Memory clean for the next fold...")
    del model, train_ds, val_ds, test_ds, predict_ds, history
    tf.keras.backend.clear_session()
    gc.collect()

np.save('/kaggle/working/MobileNetV2_5Fold_Accuracies_DualGPU.npy',  np.array(fold_accuracies))
np.save('/kaggle/working/MobileNetV2_5Fold_F1_DualGPU.npy',          np.array(fold_f1_scores))
np.save('/kaggle/working/MobileNetV2_5Fold_Precision_DualGPU.npy',   np.array(fold_precisions))
np.save('/kaggle/working/MobileNetV2_5Fold_Recall_DualGPU.npy',      np.array(fold_recalls))
np.save('/kaggle/working/MobileNetV2_5Fold_ConfMatrix_DualGPU.npy',  np.array(fold_conf_matrices))

print(f"\n🎉 ALL {K_FOLDS} FOLDS COMPLETE!")
print(f"🏆 Avg Accuracy  : {np.mean(fold_accuracies)*100:.2f}% (+/- {np.std(fold_accuracies)*100:.2f}%)")
print(f"🏆 Avg F1 Score  : {np.mean(fold_f1_scores):.4f} (+/- {np.std(fold_f1_scores):.4f})")
print(f"🏆 Avg Precision : {np.mean(fold_precisions):.4f} (+/- {np.std(fold_precisions):.4f})")
print(f"🏆 Avg Recall    : {np.mean(fold_recalls):.4f} (+/- {np.std(fold_recalls):.4f})")
print(f"⏱️ Total Experiment Time: {(time.time() - start_time) / 60:.2f} minutes")

2026-04-06 15:16:36.996018: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775488597.252563      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775488597.323817      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775488597.941636      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775488597.941682      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775488597.941686      23 computation_placer.cc:177] computation placer alr

INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')


I0000 00:00:1775488627.998544      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1775488628.004873      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


✅ Dual-GPU Strategy Initialized. Replicas: 2

Fetching Local File Paths...
✅ Successfully pooled 40384 images.
Classes found: {'MildDemented': 0, 'ModerateDemented': 1, 'NonDemented': 2, 'VeryMildDemented': 3}

🚀 STARTING 5-FOLD CROSS-VALIDATION (MobileNetV2 on T4 x2 GPUs)...

🧪 STARTING FOLD 1 / 5
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

⏳ Training Model...
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0

I0000 00:00:1775488709.273379      71 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1775488709.279844      70 cuda_dnn.cc:529] Loaded cuDNN version 91002


455/455 ━━━━━━━━━━━━━━━━━━━━ 0s 136ms/step - accuracy: 0.4096 - loss: 1.3124INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).

Epoch 1: val_accuracy improved from -inf to 0.62365, saving model to /kaggle/working/best_mobilenetv2_fold_1.keras
455/455 ━━━━━━━━━━━━━━━━━━━━ 84s 161ms/step - accuracy: 0.4098 - loss: 1.3120 - val_accuracy: 0.6236 - val_loss: 0.8956
Epoch 2/50
454/455 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step - accuracy: 0.6019 - loss: 0.9130
Epoch 2: val_accuracy improved from 0.62365 to 0.67874, saving model to /kaggle/working/best_mobilenetv2_fold_1.keras
455/455 ━━━━━━━━━━━━━━━━━━━━ 44s 94ms/step - accuracy: 0.6019 - loss: 0.9128 - val_accuracy: 0.6787 - val_loss: 0.7830
Epoch 3/50
454/455 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step - accuracy: 0.6460 - loss: 0.81